# Strands + LangSmith: Tracing, PromptHub & Evaluations

This notebook demonstrates a **Strands-native** agent (with tools) using LangSmith for:

1. **OTEL Tracing** — Strands spans exported to LangSmith via a custom exporter
2. **PromptHub** — Push agent and evaluation prompts to LangSmith Hub
3. **Datasets & Evaluators** — Create a QA dataset and wire up an LLM-judge evaluator
4. **Experiments** — Run the agent against the dataset with `client.evaluate()`
5. **Traced Agent Call** — A standalone demo showing tool use in traces

The agent (`agent.py`) has three tools: `lookup_knowledge_base`, `calculator`, and `web_search`. Model provider is configured in `setup/config.py` via `create_model()` — default is **OpenAI**, swap to **Bedrock** by commenting/uncommenting.

In [1]:
!uv sync

Resolved 172 packages in 0.79ms
Audited 166 packages in 0.03ms


## 0. Setup — Config & OTEL Tracing

Load environment variables and wire up the custom LangSmith OTEL exporter. This must run before any Strands agent calls so that spans are captured.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Wire up OTEL tracing: Strands spans → LangSmith
from langsmith_exporter import setup_langsmith_telemetry
setup_langsmith_telemetry()

from setup.config import client, DATASET_NAME, PROJECT_NAME
print(f"Project: {PROJECT_NAME}")
print(f"Dataset: {DATASET_NAME}")

Project: strands-repo-demo
Dataset: Strands Research QA


## 1. Push Prompts to LangSmith Hub

We push two prompts via the LangSmith SDK (`client.push_prompt()`):
- **`strands-research-assistant`** — the agent's system prompt (ChatPromptTemplate)
- **`strands-answer-correctness-eval`** — a StructuredPrompt for the LLM-judge evaluator, with the output schema (correctness + comment) bundled in

In [3]:
from setup.prompts import push_agent_prompt, push_eval_prompt

push_agent_prompt()
push_eval_prompt()

  Prompt 'strands-research-assistant' unchanged.


/Users/robertxu/Desktop/Projects/customer/sp/strands-repo/setup/prompts.py:148: LangChainBetaWarning: The class `StructuredPrompt` is in beta. It is actively being worked on, so the API may change.
  prompt = StructuredPrompt(


  Prompt 'strands-answer-correctness-eval' unchanged.


## 2. Create Evaluation Dataset

A simple QA dataset with 5 examples. Idempotent — skips if it already exists.

In [4]:
from setup.datasets import create_dataset

create_dataset()

  Dataset 'Strands Research QA' already exists. Skipping.


## 3. Create LLM-Judge Evaluator

Creates a **correctness** evaluator on the dataset that references the Hub eval prompt (`strands-answer-correctness-eval:latest`). This evaluator runs automatically on new experiment results.

In [5]:
from setup.evaluators import create_llm_judge_evaluator

create_llm_judge_evaluator()

  Evaluator 'correctness' already exists. Skipping.


## 4. Run Evaluation Experiment

Runs the Strands agent (defined in `agent.py`) on every dataset example via `client.evaluate()`. Each call is traced via OTEL. We also include a simple **`contains_answer`** code evaluator.

In [6]:
from agent import create_agent, ask

def run_agent(inputs: dict) -> dict:
    """Target function for client.evaluate() — invoke Strands agent, return answer."""
    agent = create_agent()
    return {"answer": ask(agent, inputs["question"])}


def contains_answer(outputs: dict, reference_outputs: dict) -> dict:
    """Code evaluator: does the output contain the reference answer?"""
    ref = reference_outputs.get("answer", "").lower().strip()
    out = outputs.get("answer", "").lower().strip()
    return {"key": "contains_answer", "score": ref in out}


results = client.evaluate(
    run_agent,
    data=DATASET_NAME,
    evaluators=[contains_answer],
    experiment_prefix="strands-research-qa",
    num_repetitions=1,
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")

View the evaluation results for experiment: 'strands-research-qa-9d738cda' at:
https://smith.langchain.com/o/58636190-0252-4526-9dc7-0b09b37b499c/datasets/3d1cede1-341d-4566-a950-cc01652de74d/compare?selectedSessions=47600600-09f6-4467-8a04-95713e15ce51




0it [00:00, ?it/s]


Tool #1: lookup_knowledge_base

Tool #1: lookup_knowledge_base

Tool #2: lookup_knowledge_base
Amazon Bedrock and OpenTelemetry serve different purposes within the tech landscape:

### Amazon Bedrock
- **StrandsDescription**: Agents supports multiple model providers, It's a fully managed including Amazon Bedrock service from Amazon that provides access, OpenAI, and Anth to foundation models fromropic. You can various leading find more AI organizations through a information on their [Git unified API.
- **FeaturesHub page](https**:
://github  -.com/strands **Multiple Model Providers-agents/sdk-python).**: Access a variety of foundation models offered by different companies.
  - **Serverless**: Operates without the need to manage servers, allowing easy scaling.
  - **Fine-tuning**: Customize models to fit specific needs.
  - **Guardrails**: Implement safety and compliance features.
  - **Knowledge Bases**: Use knowledge bases to integrate data.
- **More Info**
Tool #1: calculator
: [Amaz

## 5. Traced Agent Demo

A standalone call using `create_agent()` and `ask()` from `agent.py`. Check the LangSmith project to see the full trace.

In [7]:
agent = create_agent()

# This question exercises multiple tools: knowledge base lookup + calculator
output = ask(agent, "What is Strands Agents and what providers does it support? Also, if I run 5 agents each handling 200 requests/hour, how many requests is that per day?")
print(output)


Tool #1: lookup_knowledge_base

Tool #2: calculator
Strands Agents is an open-source SDK designed for building AI agents. It supports multiple model providers, including Amazon Bedrock, OpenAI, and Anthropic. The SDK offers features such as multi-provider support, tool use, streaming, and OpenTelemetry tracing. You can find more details on their [GitHub page](https://github.com/strands-agents/sdk-python).

If you run 5 agents, each handling 200 requests per hour, the total number of requests per day would be 24,000 requests.Strands Agents is an open-source SDK designed for building AI agents. It supports multiple model providers, including Amazon Bedrock, OpenAI, and Anthropic. The SDK offers features such as multi-provider support, tool use, streaming, and OpenTelemetry tracing. You can find more details on their [GitHub page](https://github.com/strands-agents/sdk-python).

If you run 5 agents, each handling 200 requests per hour, the total number of requests per day would be 24,000 